# Random Forest RAT classification

In [ ]:
from src.preprocessing import encode_values, convert_to_numeric
from src.utils_train_models import (load_preprocessed_dataset, 
                                    generate_outcome_training_per_z, 
                                    store_windowed_training_analysis)
from src.utils_eval import compute_statistics
from src.config import path_files
import pandas as pd
import numpy as np
import os

In [2]:
MODEL_NAME = "RandomForest"

#### Data preprocessing

In [ ]:
dataset = load_preprocessed_dataset()
 
print("Precomputed dataset:")
print(dataset)

##### Output: preprocessed dataset generation
```small_test
Generated metadata with fields: Index(['run', 'node_name', 'location', 'modem_name', 'mcc', 'country',
       'iso_code', 'rat', 'id', 'rat_name'],
      dtype='object')

Generated non metadata with fields: Index(['run', 'node_name', 'location', 'modem_name', 'mcc', 'country',
       'iso_code', 'rat', 'id', 'rat_name', 'timestamp_pck', 'target_ip_pck',
       'icmp_seq_pck', 'ttl_pck', 'rtt_ms_pck', 'timestamp_pck_loss',
       'icmp_seq_pck_loss', 'ttl_pck_loss', 'rtt_ms_pck_loss',
       'timestamp_throughput', 'tot_size_throughput', 'avg_speed_throughput',
       'tot_time_throughput', 'filesize_throughput', 'direction_throughput',
       'timestamp_current', 'tot_size_current', 'avg_speed_current',
       'tot_time_current', 'filesize_current', 'direction_current',
       'timestamp_pw_idle', 'timetamp_ms_pw_idle', 'diff_pw_idle',
       'current_pw_idle', 'voltage_pw_idle', 'timestamp_pw_upload',
       'avg_speed_pw_upload', 'tot_time_pw_upload', 'timetamp_ms_pw_upload',
       'diff_pw_upload', 'current_pw_upload', 'voltage_pw_upload'],
      dtype='object')

Average loss computed: 54.86

                 name_col   tot_row  info_lost  valid_info  percentage_loss
0           timestamp_pck  63587384   51392119    12195265            80.82
1           target_ip_pck  63587384   51392119    12195265            80.82
2            icmp_seq_pck  63587384   51392119    12195265            80.82
3                 ttl_pck  63587384   51392119    12195265            80.82
4              rtt_ms_pck  63587384   51392119    12195265            80.82
5      timestamp_pck_loss  63587384   53110235    10477149            83.52
6       icmp_seq_pck_loss  63587384   53110235    10477149            83.52
7            ttl_pck_loss  63587384   53110235    10477149            83.52
8         rtt_ms_pck_loss  63587384   53110235    10477149            83.52
9    timestamp_throughput  63587384   14688946    48898438            23.10
10    tot_size_throughput  63587384   14688946    48898438            23.10
11   avg_speed_throughput  63587384   14688946    48898438            23.10
12    tot_time_throughput  63587384   14688946    48898438            23.10
13    filesize_throughput  63587384   14688946    48898438            23.10
14   direction_throughput  63587384   14688946    48898438            23.10
15      timestamp_current  63587384   14688946    48898438            23.10
16       tot_size_current  63587384   14688946    48898438            23.10
17      avg_speed_current  63587384   14688946    48898438            23.10
18       tot_time_current  63587384   14688946    48898438            23.10
19       filesize_current  63587384   14688946    48898438            23.10
20      direction_current  63587384   14688946    48898438            23.10
21      timestamp_pw_idle  63587384   12532592    51054792            19.71
22    timetamp_ms_pw_idle  63587384   12532592    51054792            19.71
23           diff_pw_idle  63587384   12532592    51054792            19.71
24        current_pw_idle  63587384   12532592    51054792            19.71
25        voltage_pw_idle  63587384   12532592    51054792            19.71
26    timestamp_pw_upload  63587384   63252870      334514            99.47
27    avg_speed_pw_upload  63587384   63252868      334516            99.47
28     tot_time_pw_upload  63587384   63252868      334516            99.47
29  timetamp_ms_pw_upload  63587384   63252870      334514            99.47
30         diff_pw_upload  63587384   63252870      334514            99.47
31      current_pw_upload  63587384   63252870      334514            99.47
32      voltage_pw_upload  63587384   63252870      334514            99.47

nan values analysis stored in /Users/username/NDA-Lab-Project4-RAT-Classification-TL/data/analysis/nan_analysis.json

Dropped 16 columns with above-average NaN rate
Preprocessed dataset stored at: /Users/username/NDA-Lab-Project4-RAT-Classification-TL/data/outcome_preprocess/feature_dataset.csv

Precomputed dataset:
          run node_name           location    modem_name  mcc  country  \
0           1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
10          1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
57          1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
105         1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
2864        1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
...       ...       ...                ...           ...  ...      ...   
17136562    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
17140707    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
17145467    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
17149035    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
17153702    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   

         iso_code  rat     id rat_name  ...  tot_size_current  \
0              hr    0  17443       2G  ...               0.0   
10             hr    0  17558       2G  ...               0.0   
57             hr    0  17760       2G  ...               0.0   
105            hr    0  17768       2G  ...               0.0   
2864           hr    0  18830       2G  ...               0.0   
...           ...  ...    ...      ...  ...               ...   
17136562       de    9  23995   NB-IoT  ...               0.0   
17140707       de    9  24001   NB-IoT  ...               0.0   
17145467       de    9  24003   NB-IoT  ...               0.0   
17149035       de    9  24009   NB-IoT  ...               0.0   
17153702       de    9  24011   NB-IoT  ...               0.0   

          avg_speed_current  tot_time_current  filesize_current  \
0                       0.0               0.0               0.0   
10                      0.0               0.0               0.0   
57                      0.0               0.0               0.0   
105                     0.0               0.0               0.0   
2864                    0.0               0.0               0.0   
...                     ...               ...               ...   
17136562                0.0               0.0               0.0   
17140707                0.0               0.0               0.0   
17145467                0.0               0.0               0.0   
17149035                0.0               0.0               0.0   
17153702                0.0               0.0               0.0   

         direction_current timestamp_pw_idle  timetamp_ms_pw_idle  \
0                      0.0               0.0                  0.0   
10                     0.0               0.0                  0.0   
57                     0.0               0.0                  0.0   
105                    0.0               0.0                  0.0   
2864                   0.0               0.0                  0.0   
...                    ...               ...                  ...   
17136562               0.0               0.0                  0.0   
17140707               0.0               0.0                  0.0   
17145467               0.0               0.0                  0.0   
17149035               0.0               0.0                  0.0   
17153702               0.0               0.0                  0.0   

          diff_pw_idle  current_pw_idle  voltage_pw_idle  
0                  0.0              0.0              0.0  
10                 0.0              0.0              0.0  
57                 0.0              0.0              0.0  
105                0.0              0.0              0.0  
2864               0.0              0.0              0.0  
...                ...              ...              ...  
17136562           0.0              0.0              0.0  
17140707           0.0              0.0              0.0  
17145467           0.0              0.0              0.0  
17149035           0.0              0.0              0.0  
17153702           0.0              0.0              0.0  

[51060272 rows x 27 columns]
```

##### Output: preprocessed dataset reading
```small_test
Reading the content of /Users/username/NDA-Lab-Project4-RAT-Classification-TL/data/outcome_preprocess/feature_dataset.csv

Precomputed dataset:
             id  run node_name           location    modem_name  mcc  country  \
0         17443    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
1         17558    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
2         17760    1   Mark-10    Zagreb, Croatia  Quectel_BG96  219  Croatia   
3         17768    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
4         18830    1   Mark-10    Zagreb, Croatia  Quectel_EC21  219  Croatia   
...         ...  ...       ...                ...           ...  ...      ...   
51060267  23995    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
51060268  24001    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
51060269  24003    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   
51060270  24009    1    Mark-1  Würzburg, Germany  Quectel_BG96  262  Germany   
51060271  24011    1    Mark-6    Munich, Germany  Quectel_BG96  262  Germany   

         iso_code  rat rat_name  ...  tot_size_current  avg_speed_current  \
0              hr    0       2G  ...               0.0                0.0   
1              hr    0       2G  ...               0.0                0.0   
2              hr    0       2G  ...               0.0                0.0   
3              hr    0       2G  ...               0.0                0.0   
4              hr    0       2G  ...               0.0                0.0   
...           ...  ...      ...  ...               ...                ...   
51060267       de    9   NB-IoT  ...               0.0                0.0   
51060268       de    9   NB-IoT  ...               0.0                0.0   
51060269       de    9   NB-IoT  ...               0.0                0.0   
51060270       de    9   NB-IoT  ...               0.0                0.0   
51060271       de    9   NB-IoT  ...               0.0                0.0   

          tot_time_current  filesize_current direction_current  \
0                      0.0               0.0               0.0   
1                      0.0               0.0               0.0   
2                      0.0               0.0               0.0   
3                      0.0               0.0               0.0   
4                      0.0               0.0               0.0   
...                    ...               ...               ...   
51060267               0.0               0.0               0.0   
51060268               0.0               0.0               0.0   
51060269               0.0               0.0               0.0   
51060270               0.0               0.0               0.0   
51060271               0.0               0.0               0.0   

         timestamp_pw_idle  timetamp_ms_pw_idle  diff_pw_idle  \
0                      0.0                  0.0           0.0   
1                      0.0                  0.0           0.0   
2                      0.0                  0.0           0.0   
3                      0.0                  0.0           0.0   
4                      0.0                  0.0           0.0   
...                    ...                  ...           ...   
51060267               0.0                  0.0           0.0   
51060268               0.0                  0.0           0.0   
51060269               0.0                  0.0           0.0   
51060270               0.0                  0.0           0.0   
51060271               0.0                  0.0           0.0   

          current_pw_idle  voltage_pw_idle  
0                     0.0              0.0  
1                     0.0              0.0  
2                     0.0              0.0  
3                     0.0              0.0  
4                     0.0              0.0  
...                   ...              ...  
51060267              0.0              0.0  
51060268              0.0              0.0  
51060269              0.0              0.0  
51060270              0.0              0.0  
51060271              0.0              0.0  

[51060272 rows x 27 columns]
```

##### Recover the correct column data type

In [ ]:
dataset = convert_to_numeric(dataset)

list_non_numerical = []
list_numerical = []
for col in dataset.columns:
    if pd.api.types.is_numeric_dtype(dataset[col]):
        list_numerical.append(col)
    else:
        list_non_numerical.append(col)

print("Numerical column list:\n")
print(list_numerical)
print()
print("Non numerical column list:\n")
print(list_non_numerical)

##### Output:
```small_text
Numerical column list:

['run', 'mcc', 'iso_code', 'rat', 'id', 'timestamp_throughput', 'tot_size_throughput', 'avg_speed_throughput', 'tot_time_throughput', 'filesize_throughput', 'timestamp_current', 'tot_size_current', 'avg_speed_current', 'tot_time_current', 'filesize_current', 'timestamp_pw_idle', 'timetamp_ms_pw_idle', 'diff_pw_idle', 'current_pw_idle', 'voltage_pw_idle']

Non numerical column list:

['node_name', 'location', 'modem_name', 'country', 'rat_name', 'direction_throughput', 'direction_current']
```

##### RAT_NAME saved as labels

In [ ]:
RAT_NAME = [label for label in np.unique(dataset["rat_name"])]
print("RAT classification, labels name: {}".format(RAT_NAME))
dataset.drop("rat_name", axis=1, inplace=True)
list_non_numerical.remove("rat_name")

##### Output:
```small_text
RAT classification, labels name: ['2G', '3G', 'LTE CAT1', 'LTE-M', 'NB-IoT']
```

#### Dataset encoding

In [ ]:
dataset = encode_values(dataset, list_non_numerical)
dataset.to_csv(path_files["ENCODED_DATASET"], index=False)
print("Labels encoding process has been successfully completed and stored\n")
print("Encoded dataset obtained\n")
print(dataset)

##### Output:
```small_text
Column list with meaningful values of type string

['node_name', 'location', 'modem_name', 'country', 'direction_throughput', 'direction_current']
Encoding string objects within the feature dataset...

Outcome 1 step encoding

   node_name  encoded
0     Mark-1        0
1    Mark-10        1
2    Mark-11        2
3    Mark-12        3
4     Mark-2        4
5     Mark-3        5
6     Mark-4        6
7     Mark-5        7
8     Mark-6        8
9     Mark-7        9
10    Mark-9       10
Column: node_name, Number of rows with NaN value: 11868601

Outcome 2 step encoding

            location  encoded
0       Milan, Italy        0
1    Munich, Germany        1
2  Trondheim, Norway        2
3  Würzburg, Germany        3
4    Zagreb, Croatia        4
Column: location, Number of rows with NaN value: 15631980

Outcome 3 step encoding

     modem_name  encoded
0  Quectel_BG96        0
1  Quectel_EC21        1
Column: modem_name, Number of rows with NaN value: 16006451

Outcome 4 step encoding

   country  encoded
0  Croatia        0
1  Germany        1
2    Italy        2
3   Norway        3
Column: country, Number of rows with NaN value: 15824230

Outcome 5 step encoding

  direction_throughput  encoded
0                  0.0        0
1             Downlink        1
2               Uplink        2
Column: direction_throughput, Number of rows with NaN value: 24238

Outcome 6 step encoding

  direction_current  encoded
0               0.0        0
1          Downlink        1
2            Uplink        2
Column: direction_current, Number of rows with NaN value: 24238

Labels encoding process has been successfully completed and stored

Encoded dataset obtained

          run  node_name  location  modem_name  mcc  country  iso_code  rat  \
0           1          1         4           0  219        0       0.0    0   
1           1          1         4           1  219        0       0.0    0   
2           1          1         4           0  219        0       0.0    0   
3           1          1         4           1  219        0       0.0    0   
4           1          1         4           1  219        0       0.0    0   
...       ...        ...       ...         ...  ...      ...       ...  ...   
51060267    1          8         1           0  262        1       0.0    9   
51060268    1          0         3           0  262        1       0.0    9   
51060269    1          8         1           0  262        1       0.0    9   
51060270    1          0         3           0  262        1       0.0    9   
51060271    1          8         1           0  262        1       0.0    9   

             id  timestamp_throughput  ...  tot_size_current  \
0         17443                   0.0  ...               0.0   
1         17558                   0.0  ...               0.0   
2         17760                   0.0  ...               0.0   
3         17768                   0.0  ...               0.0   
4         18830                   0.0  ...               0.0   
...         ...                   ...  ...               ...   
51060267  23995                   0.0  ...               0.0   
51060268  24001                   0.0  ...               0.0   
51060269  24003                   0.0  ...               0.0   
51060270  24009                   0.0  ...               0.0   
51060271  24011                   0.0  ...               0.0   

          avg_speed_current  tot_time_current  filesize_current  \
0                       0.0               0.0               0.0   
1                       0.0               0.0               0.0   
2                       0.0               0.0               0.0   
3                       0.0               0.0               0.0   
4                       0.0               0.0               0.0   
...                     ...               ...               ...   
51060267                0.0               0.0               0.0   
51060268                0.0               0.0               0.0   
51060269                0.0               0.0               0.0   
51060270                0.0               0.0               0.0   
51060271                0.0               0.0               0.0   

          direction_current  timestamp_pw_idle  timetamp_ms_pw_idle  \
0                       0.0                0.0                  0.0   
1                       0.0                0.0                  0.0   
2                       0.0                0.0                  0.0   
3                       0.0                0.0                  0.0   
4                       0.0                0.0                  0.0   
...                     ...                ...                  ...   
51060267                NaN                0.0                  0.0   
51060268                NaN                0.0                  0.0   
51060269                NaN                0.0                  0.0   
51060270                NaN                0.0                  0.0   
51060271                NaN                0.0                  0.0   

          diff_pw_idle  current_pw_idle  voltage_pw_idle  
0                  0.0              0.0              0.0  
1                  0.0              0.0              0.0  
2                  0.0              0.0              0.0  
3                  0.0              0.0              0.0  
4                  0.0              0.0              0.0  
...                ...              ...              ...  
51060267           0.0              0.0              0.0  
51060268           0.0              0.0              0.0  
51060269           0.0              0.0              0.0  
51060270           0.0              0.0              0.0  
51060271           0.0              0.0              0.0  

[51060272 rows x 26 columns]
```

#### Generation of the windowed features dataset sorted by timestamp


In [ ]:
outcome_training_per_z = generate_outcome_training_per_z(dataset, RAT_NAME, "RandomForest")

#### Output:
```small_text
Measurement columns (10): ['id', 'tot_size_throughput', 'avg_speed_throughput', 'tot_time_throughput', 'tot_size_current', 'avg_speed_current', 'tot_time_current', 'diff_pw_idle', 'current_pw_idle', 'voltage_pw_idle']
Dataset was sorted by timestamps: ['timestamp_throughput', 'timestamp_current', 'timestamp_pw_idle']

Number of encoded groups: 85

Generation of the windowed dataset was ultimated
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 1000
100%|[32m██████████[0m| 25717625/25717625 [03:13<00:00, 132916.26it/s]
100%|[32m██████████[0m| 25342647/25342647 [03:13<00:00, 130979.45it/s]
Windowed dataset computed for z = 1000

Length dataset: 51150

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 5000
100%|[32m██████████[0m| 25717625/25717625 [01:00<00:00, 424777.68it/s]
100%|[32m██████████[0m| 25342647/25342647 [01:00<00:00, 419972.72it/s]
Windowed dataset computed for z = 5000

Length dataset: 10299

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 10000
100%|[32m██████████[0m| 25717625/25717625 [00:42<00:00, 611370.09it/s]
100%|[32m██████████[0m| 25342647/25342647 [00:42<00:00, 596845.69it/s]
Windowed dataset computed for z = 10000

Length dataset: 5190

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 50000
100%|[32m██████████[0m| 25717625/25717625 [00:36<00:00, 706553.53it/s] 
100%|[32m██████████[0m| 25342647/25342647 [00:34<00:00, 742124.22it/s] 
Windowed dataset computed for z = 50000

Length dataset: 1113

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 100000
100%|[32m██████████[0m| 25717625/25717625 [00:27<00:00, 938690.17it/s] 
100%|[32m██████████[0m| 25342647/25342647 [00:27<00:00, 917448.36it/s] 
Windowed dataset computed for z = 100000

Length dataset: 605

Training a RandomForest classifier...
Train dataset length: 25717625, test dataset length: 25342647 obtained for Z: 500000
100%|[32m██████████[0m| 25717625/25717625 [00:25<00:00, 1027199.49it/s]
100%|[32m██████████[0m| 25342647/25342647 [00:24<00:00, 1019102.10it/s]
Windowed dataset computed for z = 500000

Length dataset: 195

Training a RandomForest classifier...
```

#### Statistics computation

In [ ]:
accuracy_per_z = compute_statistics(outcome_training_per_z, RAT_NAME)

##### Output:
```small_text
------------------------------------

Results for Z: 1000

Training time[s]: 0.467876672744751

Accuracy: 0.8471733520994604

Global precision: 0.9164814052559414

Global recall: 0.8471733520994604

Global f1score: 0.8569047789499792

------------------------------------

------------------------------------

Results for Z: 5000

Training time[s]: 0.14772891998291016

Accuracy: 0.8413093415007658

Global precision: 0.9083162198364126

Global recall: 0.8413093415007658

Global f1score: 0.8499797662721285

------------------------------------

------------------------------------

Results for Z: 10000

Training time[s]: 0.10687994956970215

Accuracy: 0.8315985130111524

Global precision: 0.8953665784317072

Global recall: 0.8315985130111524

Global f1score: 0.8388389958479158

------------------------------------

------------------------------------

Results for Z: 50000

Training time[s]: 0.11036205291748047

Accuracy: 0.7734138972809668

Global precision: 0.8255597664266514

Global recall: 0.7734138972809668

Global f1score: 0.7732285055540666

------------------------------------

------------------------------------

Results for Z: 100000

Training time[s]: 0.08701515197753906

Accuracy: 0.7310513447432763

Global precision: 0.7804738470122505

Global recall: 0.7310513447432763

Global f1score: 0.72513745540618

------------------------------------

------------------------------------

Results for Z: 500000

Training time[s]: 0.08607006072998047

Accuracy: 0.6287128712871287

Global precision: 0.6853297279165419

Global recall: 0.6287128712871287

Global f1score: 0.6158046235404099

------------------------------------
```

In [9]:
max_accuracy = max(accuracy_per_z)
outcome_windowed_dataset_eval = {}
for index_row, accuracy in enumerate(accuracy_per_z):
    if accuracy == max_accuracy:
        outcome_windowed_dataset_eval["model_name"] = MODEL_NAME
        outcome_windowed_dataset_eval["window_size"] = outcome_training_per_z.loc[index_row]["z_value"]
        outcome_windowed_dataset_eval["selected_model"] = outcome_training_per_z.loc[index_row]["pretrained_model"]
        outcome_windowed_dataset_eval["windowed_dataset"] = outcome_training_per_z.loc[index_row]["features_set"]
        break
    
store_windowed_training_analysis(outcome_windowed_dataset_eval, MODEL_NAME)